In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from PIL import Image
import time

In [ ]:
# Configuración
N_COLORES = 16

def cargar_imagen(ruta):
    """Carga la imagen y la convierte a array numpy"""
    img = Image.open(ruta)
    # Convertir a RGB si es necesario (por si tiene alpha channel)
    if img.mode != 'RGB':
        img = img.convert('RGB')
    return np.array(img)

def cuantizar_colores(imagen, n_colores=16, espacio='RGB'):
    """
    Reduce la imagen a n_colores usando K-means
    
    Args:
        imagen: array numpy (H, W, 3)
        n_colores: número de colores finales
        espacio: 'RGB' o 'LAB'
    
    Returns:
        imagen_cuantizada: array con colores reducidos
        colores_paleta: array con los n colores (centroides)
        labels: etiquetas de cluster para cada pixel
    """
    print(f"Imagen original: {imagen.shape}")
    h, w, c = imagen.shape
    
    # Reshape: convertir (H, W, 3) -> (H*W, 3)
    pixels = imagen.reshape(-1, 3)
    print(f"Total de píxeles: {pixels.shape[0]:,}")
    
    # Convertir a float para K-means
    pixels_float = pixels.astype(float)
    
    # Aplicar K-means
    print(f"\nAplicando K-means con {n_colores} colores...")
    inicio = time.time()
    
    kmeans = KMeans(
        n_clusters=n_colores,
        init='k-means++',  # Mejor inicialización
        max_iter=300,
        n_init=10,  # Número de veces que se ejecuta con diferentes centroides
        random_state=42,
        verbose=0
    )
    
    labels = kmeans.fit_predict(pixels_float)
    colores_paleta = kmeans.cluster_centers_
    
    tiempo = time.time() - inicio
    print(f"K-means completado en {tiempo:.2f} segundos")
    
    # Asignar cada pixel a su centroide más cercano
    imagen_cuantizada = colores_paleta[labels]
    
    # Reshape de vuelta a imagen
    imagen_cuantizada = imagen_cuantizada.reshape(h, w, c)
    
    # Convertir de vuelta a uint8
    imagen_cuantizada = imagen_cuantizada.astype(np.uint8)
    colores_paleta = colores_paleta.astype(np.uint8)
    
    return imagen_cuantizada, colores_paleta, labels.reshape(h, w)

def visualizar_resultado(imagen_original, imagen_cuantizada, colores_paleta):
    """Visualiza la imagen original, cuantizada y la paleta de colores"""
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Imagen original
    axes[0, 0].imshow(imagen_original)
    axes[0, 0].set_title('Imagen Original', fontsize=14, fontweight='bold')
    axes[0, 0].axis('off')
    
    # Imagen cuantizada
    axes[0, 1].imshow(imagen_cuantizada)
    axes[0, 1].set_title(f'Imagen Cuantizada ({N_COLORES} colores)', fontsize=14, fontweight='bold')
    axes[0, 1].axis('off')
    
    # Paleta de colores
    paleta_visual = np.array([colores_paleta] * 50)  # Hacer más alta para visualizar
    axes[1, 0].imshow(paleta_visual)
    axes[1, 0].set_title('Paleta de Colores', fontsize=14, fontweight='bold')
    axes[1, 0].axis('off')
    
    # Información de colores (valores RGB)
    axes[1, 1].axis('off')
    texto_colores = "Colores RGB:\n\n"
    for i, color in enumerate(colores_paleta, 1):
        texto_colores += f"Color {i:2d}: RGB({color[0]:3d}, {color[1]:3d}, {color[2]:3d})\n"
    
    axes[1, 1].text(0.1, 0.9, texto_colores, 
                    fontsize=10, 
                    verticalalignment='top',
                    fontfamily='monospace')
    axes[1, 1].set_title('Valores de Colores', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

def guardar_resultado(imagen_cuantizada, ruta_salida):
    """Guarda la imagen cuantizada"""
    img_pil = Image.fromarray(imagen_cuantizada)
    img_pil.save(ruta_salida)
    print(f"\nImagen guardada en: {ruta_salida}")


In [ ]:
# =============================================================================
# EJECUCIÓN PRINCIPAL
# =============================================================================

# 1. Cargar imagen
RUTA_IMAGEN = "img/pensamientos.jpg"  # <<<< CAMBIA ESTO POR TU IMAGEN
imagen_original = cargar_imagen(RUTA_IMAGEN)

# 2. Cuantizar colores
imagen_cuantizada, colores_paleta, labels = cuantizar_colores(
    imagen_original, 
    n_colores=N_COLORES
)

# 3. Visualizar resultados
visualizar_resultado(imagen_original, imagen_cuantizada, colores_paleta)

# 4. Información adicional
print(f"\n{'='*60}")
print(f"RESUMEN DEL PASO 1")
print(f"{'='*60}")
print(f"Colores originales (aprox): {len(np.unique(imagen_original.reshape(-1, 3), axis=0)):,}")
print(f"Colores finales: {len(colores_paleta)}")
print(f"Reducción exitosa: ✓")
print(f"{'='*60}")

# 5. (Opcional) Guardar resultado
guardar_resultado(imagen_cuantizada, "paso1_cuantizada.png")

In [ ]:
labels

In [ ]:
from scipy import ndimage
from skimage import measure
import pandas as pd

In [ ]:
def segmentar_regiones(imagen_cuantizada, labels, colores_paleta):
    """
    Identifica regiones conectadas (connected components) para cada color
    
    Args:
        imagen_cuantizada: imagen con colores reducidos
        labels: etiquetas de cluster de K-means (H, W)
        colores_paleta: array con los 20 colores
    
    Returns:
        mapa_regiones: array donde cada región tiene un ID único
        info_regiones: DataFrame con información de cada región
    """
    h, w = labels.shape
    print(f"Procesando imagen de {h}x{w} píxeles...")
    
    # Crear mapa de regiones (inicialmente vacío)
    mapa_regiones = np.zeros((h, w), dtype=np.int32)
    
    # Lista para almacenar información de cada región
    regiones_info = []
    region_id_global = 1  # Contador global de regiones
    
    # Procesar cada color por separado
    print(f"\nBuscando regiones conectadas por color...")
    
    for color_idx in range(len(colores_paleta)):
        # Crear máscara binaria para este color
        mascara_color = (labels == color_idx).astype(np.uint8)
        
        # Encontrar componentes conectados (8-connectivity)
        regiones_etiquetadas, num_regiones = ndimage.label(mascara_color)
        
        if num_regiones == 0:
            continue
        
        print(f"  Color {color_idx + 1:2d} RGB{tuple(colores_paleta[color_idx])}: {num_regiones:3d} regiones")
        
        # Para cada región de este color
        for region_idx in range(1, num_regiones + 1):
            mascara_region = (regiones_etiquetadas == region_idx)
            area = np.sum(mascara_region)
            
            # Asignar ID global único a esta región
            mapa_regiones[mascara_region] = region_id_global
            
            # Guardar información
            regiones_info.append({
                'region_id': region_id_global,
                'color_id': color_idx + 1,  # 1-indexed para el usuario
                'color_rgb': tuple(colores_paleta[color_idx]),
                'area_pixels': area,
                'porcentaje': 100 * area / (h * w)
            })
            
            region_id_global += 1
    
    # Convertir a DataFrame
    df_regiones = pd.DataFrame(regiones_info)
    
    print(f"\n{'='*60}")
    print(f"Total de regiones encontradas: {len(df_regiones)}")
    print(f"{'='*60}")
    
    return mapa_regiones, df_regiones

def visualizar_regiones(imagen_original, imagen_cuantizada, mapa_regiones, df_regiones):
    """Visualiza las regiones segmentadas"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    
    # Imagen original
    axes[0, 0].imshow(imagen_original)
    axes[0, 0].set_title('Imagen Original', fontsize=14, fontweight='bold')
    axes[0, 0].axis('off')
    
    # Imagen cuantizada
    axes[0, 1].imshow(imagen_cuantizada)
    axes[0, 1].set_title(f'Imagen Cuantizada ({N_COLORES} colores)', fontsize=14, fontweight='bold')
    axes[0, 1].axis('off')
    
    # Mapa de regiones (cada región con color aleatorio para distinguir)
    # Usar colormap para visualizar mejor
    regiones_visuales = mapa_regiones.copy()
    axes[1, 0].imshow(regiones_visuales, cmap='nipy_spectral', interpolation='nearest')
    axes[1, 0].set_title(f'Regiones Segmentadas ({len(df_regiones)} regiones)', 
                         fontsize=14, fontweight='bold')
    axes[1, 0].axis('off')
    
    # Imagen cuantizada con bordes de regiones
    from skimage.segmentation import find_boundaries
    bordes = find_boundaries(mapa_regiones, mode='thick')
    
    img_con_bordes = imagen_cuantizada.copy()
    img_con_bordes[bordes] = [0, 0, 0]  # Bordes negros
    
    axes[1, 1].imshow(img_con_bordes)
    axes[1, 1].set_title('Regiones con Bordes Delimitados', fontsize=14, fontweight='bold')
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.show()

def analizar_regiones(df_regiones, n_colores=20):
    """Muestra estadísticas detalladas de las regiones"""
    
    print(f"\n{'='*70}")
    print(f"ANÁLISIS DE REGIONES")
    print(f"{'='*70}\n")
    
    # Resumen por color
    print("Distribución de regiones por color:")
    print("-" * 70)
    resumen_colores = df_regiones.groupby('color_id').agg({
        'region_id': 'count',
        'area_pixels': ['sum', 'mean', 'min', 'max']
    }).round(1)
    resumen_colores.columns = ['Num_Regiones', 'Area_Total', 'Area_Promedio', 'Area_Min', 'Area_Max']
    print(resumen_colores)
    
    # Regiones más grandes y más pequeñas
    print(f"\n{'='*70}")
    print("Top 10 regiones MÁS GRANDES:")
    print("-" * 70)
    top_grandes = df_regiones.nlargest(10, 'area_pixels')[['region_id', 'color_id', 'color_rgb', 'area_pixels', 'porcentaje']]
    print(top_grandes.to_string(index=False))
    
    print(f"\n{'='*70}")
    print("Top 10 regiones MÁS PEQUEÑAS:")
    print("-" * 70)
    top_pequenas = df_regiones.nsmallest(10, 'area_pixels')[['region_id', 'color_id', 'color_rgb', 'area_pixels', 'porcentaje']]
    print(top_pequenas.to_string(index=False))
    
    # Estadísticas generales
    print(f"\n{'='*70}")
    print("ESTADÍSTICAS GENERALES:")
    print("-" * 70)
    print(f"Total de regiones: {len(df_regiones)}")
    print(f"Área promedio: {df_regiones['area_pixels'].mean():.1f} píxeles")
    print(f"Área mediana: {df_regiones['area_pixels'].median():.1f} píxeles")
    print(f"Área mínima: {df_regiones['area_pixels'].min()} píxeles")
    print(f"Área máxima: {df_regiones['area_pixels'].max()} píxeles")
    print(f"Desviación estándar: {df_regiones['area_pixels'].std():.1f} píxeles")
    print(f"{'='*70}\n")
    
    # Visualizar distribución de tamaños
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histograma de tamaños
    axes[0].hist(df_regiones['area_pixels'], bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Área (píxeles)', fontsize=12)
    axes[0].set_ylabel('Frecuencia', fontsize=12)
    axes[0].set_title('Distribución de Tamaños de Regiones', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    
    # Barras de regiones por color
    regiones_por_color = df_regiones.groupby('color_id').size()
    axes[1].bar(regiones_por_color.index, regiones_por_color.values, edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('Color ID', fontsize=12)
    axes[1].set_ylabel('Número de Regiones', fontsize=12)
    axes[1].set_title('Regiones por Color', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()



In [ ]:
# =============================================================================
# EJECUCIÓN DEL PASO 2
# =============================================================================

print("="*70)
print("PASO 2: SEGMENTACIÓN POR CONNECTED COMPONENTS")
print("="*70)

# Segmentar regiones
mapa_regiones, df_regiones = segmentar_regiones(imagen_cuantizada, labels, colores_paleta)

# Visualizar resultados
visualizar_regiones(imagen_original, imagen_cuantizada, mapa_regiones, df_regiones)

# Análisis detallado
analizar_regiones(df_regiones, N_COLORES)

In [ ]:
df_regiones.describe()
# df_regiones[df_regiones['area_pixels'] < 1000]

In [ ]:
df_regiones[df_regiones['area_pixels'] <= 100].describe()

In [ ]:
df_regiones_filtrado = df_regiones[df_regiones['area_pixels'] <= 100]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_regiones_filtrado['area_pixels'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Área (píxeles)', fontsize=12)
axes[0].set_ylabel('Frecuencia', fontsize=12)
axes[0].set_title('Distribución de Tamaños de Regiones', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
from scipy.ndimage import distance_transform_edt
from collections import defaultdict

In [ ]:
def calcular_area_minima_sugerida(imagen_shape, porcentaje=0.1):
    """
    Calcula un área mínima sugerida basada en el tamaño de la imagen
    
    Args:
        imagen_shape: (height, width) de la imagen
        porcentaje: porcentaje del área total (default 0.1%)
    
    Returns:
        area_minima en píxeles
    """
    h, w = imagen_shape[:2]
    area_total = h * w
    area_minima = int(area_total * porcentaje / 100)
    return max(area_minima, 50)  # Mínimo absoluto de 50 píxeles

def encontrar_vecino_mas_cercano(mapa_regiones, region_id, mascara_region, df_regiones):
    """
    Encuentra la región vecina más grande (preferiblemente del mismo color)
    
    Args:
        mapa_regiones: mapa de regiones actual
        region_id: ID de la región a fusionar
        mascara_region: máscara binaria de la región
        df_regiones: DataFrame con info de regiones
    
    Returns:
        ID de la región vecina más apropiada
    """
    # Dilatar la región para encontrar vecinos
    from scipy.ndimage import binary_dilation
    
    estructura = np.ones((3, 3))  # 8-connectivity
    mascara_dilatada = binary_dilation(mascara_region, structure=estructura)
    
    # Encontrar regiones vecinas (en el borde)
    borde = mascara_dilatada & ~mascara_region
    regiones_vecinas = np.unique(mapa_regiones[borde])
    regiones_vecinas = regiones_vecinas[regiones_vecinas != 0]  # Excluir background
    regiones_vecinas = regiones_vecinas[regiones_vecinas != region_id]  # Excluir a sí misma
    
    if len(regiones_vecinas) == 0:
        return None
    
    # Obtener info de la región actual
    color_actual = df_regiones[df_regiones['region_id'] == region_id]['color_id'].values[0]
    
    # Priorizar vecinos del mismo color, luego por tamaño
    vecinos_info = df_regiones[df_regiones['region_id'].isin(regiones_vecinas)].copy()
    
    # Score: mismo color = +1000000, luego por área
    vecinos_info['score'] = vecinos_info['area_pixels'].copy()
    vecinos_info.loc[vecinos_info['color_id'] == color_actual, 'score'] += 1000000
    
    mejor_vecino = vecinos_info.loc[vecinos_info['score'].idxmax(), 'region_id']
    
    return mejor_vecino

def filtrar_regiones_pequenas(mapa_regiones, df_regiones, area_minima):
    """
    Elimina regiones pequeñas fusionándolas con sus vecinos
    
    Args:
        mapa_regiones: mapa de regiones actual
        df_regiones: DataFrame con info de regiones
        area_minima: umbral de área mínima en píxeles
    
    Returns:
        mapa_regiones_limpio: mapa actualizado
        df_regiones_limpio: DataFrame actualizado
        estadisticas: dict con stats del proceso
    """
    print(f"\n{'='*70}")
    print(f"FILTRANDO REGIONES MENORES A {area_minima} PÍXELES")
    print(f"{'='*70}\n")
    
    # Identificar regiones pequeñas
    regiones_pequenas = df_regiones[df_regiones['area_pixels'] < area_minima].copy()
    regiones_grandes = df_regiones[df_regiones['area_pixels'] >= area_minima].copy()
    
    print(f"Regiones pequeñas a eliminar: {len(regiones_pequenas)}")
    print(f"Regiones que se mantienen: {len(regiones_grandes)}")
    
    if len(regiones_pequenas) == 0:
        print("\n¡No hay regiones pequeñas que filtrar!")
        return mapa_regiones, df_regiones, {'eliminadas': 0, 'fusionadas': 0}
    
    # Ordenar por tamaño (procesar las más pequeñas primero)
    regiones_pequenas = regiones_pequenas.sort_values('area_pixels')
    
    # Crear copia del mapa para modificar
    mapa_limpio = mapa_regiones.copy()
    
    # Tracking de fusiones
    fusiones = {}  # region_pequena -> region_destino
    regiones_eliminadas = 0
    
    print("\nProcesando fusiones...")
    for idx, row in regiones_pequenas.iterrows():
        region_id = row['region_id']
        
        # Crear máscara de esta región
        mascara_region = (mapa_regiones == region_id)
        
        if not mascara_region.any():
            continue
        
        # Encontrar vecino más apropiado
        vecino_id = encontrar_vecino_mas_cercano(mapa_limpio, region_id, mascara_region, df_regiones)
        
        if vecino_id is None:
            print(f"  ⚠ Región {region_id} sin vecinos, se mantiene")
            continue
        
        # Fusionar: reasignar píxeles de region_id a vecino_id
        mapa_limpio[mapa_limpio == region_id] = vecino_id
        fusiones[region_id] = vecino_id
        regiones_eliminadas += 1
        
        if regiones_eliminadas % 50 == 0:
            print(f"  Procesadas {regiones_eliminadas} fusiones...")
    
    print(f"\n✓ Fusiones completadas: {regiones_eliminadas}")
    
    # Reconstruir DataFrame de regiones
    print("\nRecalculando áreas de regiones...")
    nuevas_regiones = []
    
    for region_id in np.unique(mapa_limpio):
        if region_id == 0:
            continue
        
        mascara = (mapa_limpio == region_id)
        area = np.sum(mascara)
        
        # Buscar color original de esta región
        info_original = df_regiones[df_regiones['region_id'] == region_id]
        
        if len(info_original) > 0:
            color_id = info_original.iloc[0]['color_id']
            color_rgb = info_original.iloc[0]['color_rgb']
        else:
            # Esta región fue destino de fusión, mantener su info
            continue
        
        nuevas_regiones.append({
            'region_id': region_id,
            'color_id': color_id,
            'color_rgb': color_rgb,
            'area_pixels': area,
            'porcentaje': 100 * area / mapa_limpio.size
        })
    
    df_limpio = pd.DataFrame(nuevas_regiones)
    
    # Estadísticas
    stats = {
        'eliminadas': regiones_eliminadas,
        'fusionadas': len(fusiones),
        'antes': len(df_regiones),
        'despues': len(df_limpio),
        'reduccion_porcentaje': 100 * (len(df_regiones) - len(df_limpio)) / len(df_regiones)
    }
    
    print(f"\n{'='*70}")
    print(f"RESUMEN DE FILTRADO")
    print(f"{'='*70}")
    print(f"Regiones antes: {stats['antes']}")
    print(f"Regiones después: {stats['despues']}")
    print(f"Reducción: {stats['reduccion_porcentaje']:.1f}%")
    print(f"{'='*70}\n")
    
    return mapa_limpio, df_limpio, stats

def comparar_antes_despues(imagen_cuantizada, mapa_antes, mapa_despues, df_antes, df_despues):
    """Visualiza comparación antes/después del filtrado"""
    
    from skimage.segmentation import find_boundaries
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    
    # ANTES - Mapa de regiones
    axes[0, 0].imshow(mapa_antes, cmap='nipy_spectral', interpolation='nearest')
    axes[0, 0].set_title(f'ANTES: {len(df_antes)} regiones', fontsize=14, fontweight='bold')
    axes[0, 0].axis('off')
    
    # ANTES - Con bordes
    bordes_antes = find_boundaries(mapa_antes, mode='thick')
    img_bordes_antes = imagen_cuantizada.copy()
    img_bordes_antes[bordes_antes] = [0, 0, 0]
    
    axes[0, 1].imshow(img_bordes_antes)
    axes[0, 1].set_title('ANTES: Con Bordes', fontsize=14, fontweight='bold')
    axes[0, 1].axis('off')
    
    # DESPUÉS - Mapa de regiones
    axes[1, 0].imshow(mapa_despues, cmap='nipy_spectral', interpolation='nearest')
    axes[1, 0].set_title(f'DESPUÉS: {len(df_despues)} regiones', fontsize=14, fontweight='bold')
    axes[1, 0].axis('off')
    
    # DESPUÉS - Con bordes
    bordes_despues = find_boundaries(mapa_despues, mode='thick')
    img_bordes_despues = imagen_cuantizada.copy()
    img_bordes_despues[bordes_despues] = [0, 0, 0]
    
    axes[1, 1].imshow(img_bordes_despues)
    axes[1, 1].set_title('DESPUÉS: Con Bordes (Limpio)', fontsize=14, fontweight='bold')
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Gráfico de distribución de tamaños
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    
    ax.hist(df_antes['area_pixels'], bins=50, alpha=0.5, label='Antes', edgecolor='black')
    ax.hist(df_despues['area_pixels'], bins=50, alpha=0.5, label='Después', edgecolor='black')
    ax.axvline(area_minima_sugerida, color='red', linestyle='--', linewidth=2, label=f'Umbral: {area_minima_sugerida} px')
    ax.set_xlabel('Área (píxeles)', fontsize=12)
    ax.set_ylabel('Frecuencia', fontsize=12)
    ax.set_title('Comparación de Distribución de Tamaños', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


In [ ]:

# =============================================================================
# EJECUCIÓN DEL PASO 3
# =============================================================================

print("="*70)
print("PASO 3: FILTRAR REGIONES PEQUEÑAS")
print("="*70)

# Calcular área mínima sugerida (puedes ajustar el porcentaje)
area_minima_sugerida = calcular_area_minima_sugerida(
    imagen_original.shape, 
    porcentaje=0.15  # 0.15% del área total - AJUSTA ESTE VALOR
)

print(f"\nÁrea mínima sugerida: {area_minima_sugerida} píxeles")
print(f"(Puedes modificar este valor si quieres regiones más grandes o pequeñas)\n")

# Opción: usar valor personalizado
# AREA_MINIMA = 200  # Descomentar para usar valor fijo

AREA_MINIMA = area_minima_sugerida

# Filtrar regiones pequeñas
mapa_regiones_limpio, df_regiones_limpio, stats_filtrado = filtrar_regiones_pequenas(
    mapa_regiones, 
    df_regiones, 
    AREA_MINIMA
)

# Visualizar comparación
comparar_antes_despues(
    imagen_cuantizada,
    mapa_regiones, 
    mapa_regiones_limpio,
    df_regiones,
    df_regiones_limpio
)

# Análisis del resultado limpio
print("\n" + "="*70)
print("ANÁLISIS POST-FILTRADO")
print("="*70 + "\n")

analizar_regiones(df_regiones_limpio, N_COLORES)

In [ ]:
from scipy.ndimage import median_filter, binary_closing, binary_opening
from skimage.morphology import disk, remove_small_holes

In [ ]:
def suavizar_bordes(mapa_regiones, df_regiones, kernel_size=3, aplicar_morfologia=True):
    """
    Suaviza los bordes de las regiones usando filtros
    
    Args:
        mapa_regiones: mapa de regiones actual
        df_regiones: DataFrame con info de regiones
        kernel_size: tamaño del kernel para median filter (3, 5, 7, etc.)
        aplicar_morfologia: si aplicar operaciones morfológicas adicionales
    
    Returns:
        mapa_suavizado: mapa con bordes suavizados
        df_actualizado: DataFrame actualizado
        stats: estadísticas del proceso
    """
    print(f"\n{'='*70}")
    print(f"SUAVIZANDO BORDES (kernel size: {kernel_size})")
    print(f"{'='*70}\n")
    
    h, w = mapa_regiones.shape
    
    # Paso 1: Aplicar median filter
    print("Aplicando filtro de mediana...")
    mapa_suavizado = median_filter(mapa_regiones, size=kernel_size)
    
    # Paso 2: Operaciones morfológicas por región (opcional pero recomendado)
    if aplicar_morfologia:
        print("Aplicando operaciones morfológicas para limpiar artefactos...")
        
        mapa_morfologico = np.zeros_like(mapa_suavizado)
        
        # Procesar cada región individualmente
        regiones_unicas = np.unique(mapa_suavizado)
        regiones_unicas = regiones_unicas[regiones_unicas != 0]
        
        for i, region_id in enumerate(regiones_unicas):
            if (i + 1) % 20 == 0:
                print(f"  Procesadas {i + 1}/{len(regiones_unicas)} regiones...")
            
            # Máscara binaria de esta región
            mascara = (mapa_suavizado == region_id)
            
            # Cerrar pequeños huecos (closing)
            mascara_cerrada = binary_closing(mascara, structure=disk(2))
            
            # Rellenar huecos internos
            mascara_rellena = remove_small_holes(mascara_cerrada, area_threshold=50)
            
            # Asignar a mapa final
            mapa_morfologico[mascara_rellena] = region_id
        
        mapa_suavizado = mapa_morfologico
        print(f"  ✓ {len(regiones_unicas)} regiones procesadas")
    
    # Paso 3: Re-etiquetar regiones (por si alguna se fragmentó)
    print("\nRe-etiquetando regiones...")
    mapa_final = np.zeros_like(mapa_suavizado)
    nuevas_regiones = []
    nuevo_id = 1
    
    # Diccionario para mapear color_id original
    color_map = df_regiones.set_index('region_id')['color_id'].to_dict()
    rgb_map = df_regiones.set_index('region_id')['color_rgb'].to_dict()
    
    regiones_procesadas = set()
    
    for region_id_original in np.unique(mapa_suavizado):
        if region_id_original == 0 or region_id_original in regiones_procesadas:
            continue
        
        # Encontrar componentes conectados de esta región
        mascara = (mapa_suavizado == region_id_original)
        componentes, num_componentes = ndimage.label(mascara)
        
        # Obtener color original
        color_id = color_map.get(region_id_original, 1)
        color_rgb = rgb_map.get(region_id_original, (0, 0, 0))
        
        # Re-etiquetar cada componente
        for comp_idx in range(1, num_componentes + 1):
            mascara_comp = (componentes == comp_idx)
            area = np.sum(mascara_comp)
            
            if area > 10:  # Ignorar fragmentos muy pequeños
                mapa_final[mascara_comp] = nuevo_id
                
                nuevas_regiones.append({
                    'region_id': nuevo_id,
                    'color_id': color_id,
                    'color_rgb': color_rgb,
                    'area_pixels': area,
                    'porcentaje': 100 * area / (h * w)
                })
                
                nuevo_id += 1
        
        regiones_procesadas.add(region_id_original)
    
    df_actualizado = pd.DataFrame(nuevas_regiones)
    
    # Estadísticas
    stats = {
        'regiones_antes': len(df_regiones),
        'regiones_despues': len(df_actualizado),
        'fragmentacion': len(df_actualizado) - len(df_regiones),
        'kernel_size': kernel_size,
        'morfologia': aplicar_morfologia
    }
    
    print(f"\n{'='*70}")
    print(f"RESULTADO DEL SUAVIZADO")
    print(f"{'='*70}")
    print(f"Regiones antes: {stats['regiones_antes']}")
    print(f"Regiones después: {stats['regiones_despues']}")
    
    if stats['fragmentacion'] > 0:
        print(f"⚠ Fragmentación detectada: +{stats['fragmentacion']} regiones")
        print(f"  (Algunas regiones se dividieron durante el suavizado)")
    elif stats['fragmentacion'] < 0:
        print(f"✓ Fusión detectada: {abs(stats['fragmentacion'])} regiones menos")
    else:
        print(f"✓ Número de regiones se mantuvo estable")
    
    print(f"{'='*70}\n")
    
    return mapa_final, df_actualizado, stats

def visualizar_suavizado(imagen_cuantizada, mapa_antes, mapa_despues, df_antes, df_despues):
    """Visualiza el efecto del suavizado en los bordes"""
    
    from skimage.segmentation import find_boundaries
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # ANTES - Bordes
    bordes_antes = find_boundaries(mapa_antes, mode='thick')
    img_bordes_antes = imagen_cuantizada.copy()
    img_bordes_antes[bordes_antes] = [255, 0, 0]  # Rojo
    
    axes[0, 0].imshow(img_bordes_antes)
    axes[0, 0].set_title('ANTES: Bordes', fontsize=13, fontweight='bold')
    axes[0, 0].axis('off')
    
    # ANTES - Zoom de una sección
    h, w = mapa_antes.shape
    y1, y2 = h//3, 2*h//3
    x1, x2 = w//3, 2*w//3
    
    axes[0, 1].imshow(img_bordes_antes[y1:y2, x1:x2])
    axes[0, 1].set_title('ANTES: Zoom (detalle)', fontsize=13, fontweight='bold')
    axes[0, 1].axis('off')
    
    # ANTES - Solo bordes en negro
    solo_bordes_antes = np.ones_like(imagen_cuantizada) * 255
    solo_bordes_antes[bordes_antes] = [0, 0, 0]
    
    axes[0, 2].imshow(solo_bordes_antes)
    axes[0, 2].set_title('ANTES: Contornos aislados', fontsize=13, fontweight='bold')
    axes[0, 2].axis('off')
    
    # DESPUÉS - Bordes
    bordes_despues = find_boundaries(mapa_despues, mode='thick')
    img_bordes_despues = imagen_cuantizada.copy()
    img_bordes_despues[bordes_despues] = [0, 255, 0]  # Verde
    
    axes[1, 0].imshow(img_bordes_despues)
    axes[1, 0].set_title('DESPUÉS: Bordes Suavizados', fontsize=13, fontweight='bold')
    axes[1, 0].axis('off')
    
    # DESPUÉS - Zoom
    axes[1, 1].imshow(img_bordes_despues[y1:y2, x1:x2])
    axes[1, 1].set_title('DESPUÉS: Zoom (detalle)', fontsize=13, fontweight='bold')
    axes[1, 1].axis('off')
    
    # DESPUÉS - Solo bordes en negro
    solo_bordes_despues = np.ones_like(imagen_cuantizada) * 255
    solo_bordes_despues[bordes_despues] = [0, 0, 0]
    
    axes[1, 2].imshow(solo_bordes_despues)
    axes[1, 2].set_title('DESPUÉS: Contornos suavizados', fontsize=13, fontweight='bold')
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Comparación de complejidad de bordes
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    
    complejidad_antes = np.sum(bordes_antes)
    complejidad_despues = np.sum(bordes_despues)
    
    ax.bar(['Antes', 'Después'], 
           [complejidad_antes, complejidad_despues],
           color=['red', 'green'], alpha=0.7, edgecolor='black', linewidth=2)
    ax.set_ylabel('Píxeles de borde', fontsize=12)
    ax.set_title('Complejidad de Bordes (menos = más suave)', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Añadir valores
    for i, v in enumerate([complejidad_antes, complejidad_despues]):
        ax.text(i, v + complejidad_antes*0.02, f'{v:,}', ha='center', fontsize=11, fontweight='bold')
    
    reduccion = 100 * (complejidad_antes - complejidad_despues) / complejidad_antes
    ax.text(0.5, complejidad_antes * 0.5, f'Reducción: {reduccion:.1f}%', 
            ha='center', fontsize=13, bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))
    
    plt.tight_layout()
    plt.show()


In [ ]:
# =============================================================================
# EJECUCIÓN DEL PASO 4
# =============================================================================

print("="*70)
print("PASO 4: SUAVIZADO DE BORDES")
print("="*70)

# Parámetros ajustables
KERNEL_SIZE = 5  # 3, 5, 7... (impar) - Mayor = más suavizado
APLICAR_MORFOLOGIA = True  # True = limpia mejor, False = más rápido

print(f"\nParámetros:")
print(f"  - Kernel size: {KERNEL_SIZE}")
print(f"  - Morfología: {'Sí' if APLICAR_MORFOLOGIA else 'No'}")

# Suavizar bordes
mapa_regiones_suavizado, df_regiones_suavizado, stats_suavizado = suavizar_bordes(
    mapa_regiones_limpio,
    df_regiones_limpio,
    kernel_size=KERNEL_SIZE,
    aplicar_morfologia=APLICAR_MORFOLOGIA
)

# Visualizar comparación
visualizar_suavizado(
    imagen_cuantizada,
    mapa_regiones_limpio,
    mapa_regiones_suavizado,
    df_regiones_limpio,
    df_regiones_suavizado
)

# Análisis actualizado
print("\n" + "="*70)
print("ANÁLISIS POST-SUAVIZADO")
print("="*70 + "\n")

analizar_regiones(df_regiones_suavizado, N_COLORES)

print("\n✓ Paso 4 completado - Bordes suavizados")

In [ ]:
import cv2
from PIL import Image, ImageDraw, ImageFont

In [ ]:
def generar_imagen_para_pintar(mapa_regiones, df_regiones, colores_paleta, 
                                grosor_borde=1, tamano_numero='auto',
                                fondo_blanco=True):
    """
    Genera la imagen final tipo "pintar por números"
    
    Args:
        mapa_regiones: mapa de regiones final
        df_regiones: DataFrame con info de regiones
        colores_paleta: array con los 20 colores
        grosor_borde: grosor de los bordes en píxeles
        tamano_numero: tamaño de fuente ('auto' o int)
        fondo_blanco: si True, fondo blanco; si False, mantiene colores tenues
    
    Returns:
        img_para_pintar: imagen con bordes y números
        img_solucion: imagen con colores (referencia)
        paleta_numerada: imagen de la paleta con números
    """
    print(f"\n{'='*70}")
    print(f"GENERANDO IMAGEN FINAL PARA PINTAR")
    print(f"{'='*70}\n")
    
    h, w = mapa_regiones.shape
    
    # 1. Crear imagen solución (con colores)
    print("Generando imagen solución...")
    img_solucion = np.zeros((h, w, 3), dtype=np.uint8)
    
    for _, row in df_regiones.iterrows():
        region_id = row['region_id']
        color_rgb = row['color_rgb']
        mascara = (mapa_regiones == region_id)
        img_solucion[mascara] = color_rgb
    
    # 2. Crear imagen base para pintar
    print("Generando imagen base para pintar...")
    if fondo_blanco:
        img_base = np.ones((h, w, 3), dtype=np.uint8) * 255  # Blanco
    else:
        # Colores muy tenues (30% de saturación)
        img_base = (img_solucion * 0.3 + 255 * 0.7).astype(np.uint8)
    
    # 3. Detectar y dibujar contornos
    print("Detectando contornos...")
    from skimage.segmentation import find_boundaries
    
    # Encontrar todos los bordes
    bordes = find_boundaries(mapa_regiones, mode='thick')
    
    # Dilatar bordes para hacerlos más gruesos
    if grosor_borde > 1:
        from scipy.ndimage import binary_dilation
        estructura = np.ones((grosor_borde, grosor_borde))
        bordes = binary_dilation(bordes, structure=estructura)
    
    # Dibujar bordes en negro
    img_base[bordes] = [0, 0, 0]
    
    # 4. Calcular centroides y colocar números
    print("Calculando centroides y colocando números...")
    
    # Convertir a PIL para dibujar texto
    img_pil = Image.fromarray(img_base)
    draw = ImageDraw.Draw(img_pil)
    
    # Determinar tamaño de fuente automáticamente si es necesario
    if tamano_numero == 'auto':
        # Basado en el área promedio de las regiones
        area_promedio = df_regiones['area_pixels'].median()
        tamano_fuente = max(12, min(60, int(np.sqrt(area_promedio) / 3)))
    else:
        tamano_fuente = tamano_numero
    
    print(f"Tamaño de fuente: {tamano_fuente}")
    
    # Intentar cargar fuente, usar default si falla
    try:
        fuente = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", tamano_fuente)
    except:
        try:
            fuente = ImageFont.truetype("arial.ttf", tamano_fuente)
        except:
            fuente = ImageFont.load_default()
            print("⚠ Usando fuente por defecto (puede verse pequeña)")
    
    # Mapear region_id a número secuencial por color
    print("Asignando números a regiones...")
    df_regiones_sorted = df_regiones.sort_values(['color_id', 'area_pixels'], ascending=[True, False])
    df_regiones_sorted['numero'] = df_regiones_sorted.groupby('color_id').cumcount() + 1
    
    # Crear diccionario region_id -> (color_id, numero_en_color)
    numeros_map = {}
    for _, row in df_regiones_sorted.iterrows():
        numeros_map[row['region_id']] = (row['color_id'], row['numero'])
    
    # Colocar números en centroides
    regiones_numeradas = 0
    
    for region_id in np.unique(mapa_regiones):
        if region_id == 0:
            continue
        
        mascara = (mapa_regiones == region_id)
        
        # Calcular centroide
        coords = np.argwhere(mascara)
        if len(coords) == 0:
            continue
        
        centroide_y = int(coords[:, 0].mean())
        centroide_x = int(coords[:, 1].mean())
        
        # Obtener número para esta región
        color_id, numero = numeros_map.get(region_id, (0, 0))
        
        # Texto a mostrar: solo el color_id (más simple)
        texto = str(color_id)
        
        # Obtener tamaño del texto
        bbox = draw.textbbox((0, 0), texto, font=fuente)
        texto_w = bbox[2] - bbox[0]
        texto_h = bbox[3] - bbox[1]
        
        # Posición centrada
        pos_x = centroide_x - texto_w // 2
        pos_y = centroide_y - texto_h // 2
        
        # Dibujar fondo blanco para el número (mejor legibilidad)
        # padding = 3
        # draw.rectangle(
        #     [pos_x - padding, pos_y - padding, 
        #      pos_x + texto_w + padding, pos_y + texto_h + padding],
        #     fill=(255, 255, 255)
        # )
        
        # Dibujar número en negro
        draw.text((pos_x, pos_y), texto, fill=(0, 0, 0), font=fuente)
        
        regiones_numeradas += 1
    
    print(f"✓ {regiones_numeradas} regiones numeradas")
    
    # Convertir de vuelta a numpy
    img_para_pintar = np.array(img_pil)
    
    # 5. Generar paleta numerada
    print("Generando paleta de colores...")
    paleta_numerada = generar_paleta_numerada(colores_paleta, df_regiones)
    
    print(f"\n{'='*70}")
    print("✓ IMAGEN PARA PINTAR COMPLETADA")
    print(f"{'='*70}\n")
    
    return img_para_pintar, img_solucion, paleta_numerada

def generar_paleta_numerada(colores_paleta, df_regiones):
    """
    Genera una imagen de paleta con cada color y su número
    """
    n_colores = len(colores_paleta)
    
    # Dimensiones
    ancho_cuadro = 150
    alto_cuadro = 80
    cols = 4
    rows = (n_colores + cols - 1) // cols
    
    ancho_total = ancho_cuadro * cols + 40
    alto_total = alto_cuadro * rows + 80
    
    # Crear imagen
    img = Image.new('RGB', (ancho_total, alto_total), (255, 255, 255))
    draw = ImageDraw.Draw(img)
    
    # Título
    try:
        fuente_titulo = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 24)
        fuente_numero = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 32)
        fuente_rgb = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 12)
    except:
        fuente_titulo = ImageFont.load_default()
        fuente_numero = ImageFont.load_default()
        fuente_rgb = ImageFont.load_default()
    
    draw.text((20, 20), "PALETA DE COLORES", fill=(0, 0, 0), font=fuente_titulo)
    
    # Dibujar cada color
    y_offset = 70
    
    for i, color in enumerate(colores_paleta):
        row = i // cols
        col = i % cols
        
        x = 20 + col * ancho_cuadro
        y = y_offset + row * alto_cuadro
        
        # Cuadro de color
        draw.rectangle([x, y, x + 60, y + 60], fill=tuple(color), outline=(0, 0, 0), width=2)
        
        # Número
        draw.text((x + 70, y + 5), f"#{i+1}", fill=(0, 0, 0), font=fuente_numero)
        
        # Valores RGB
        rgb_text = f"RGB({color[0]}, {color[1]}, {color[2]})"
        draw.text((x + 70, y + 45), rgb_text, fill=(100, 100, 100), font=fuente_rgb)
    
    return np.array(img)

def visualizar_resultado_final(img_original, img_para_pintar, img_solucion, paleta_numerada):
    """Visualiza el resultado final completo"""
    
    fig = plt.figure(figsize=(18, 14))
    gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 0.7])
    
    # Original
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.imshow(img_original)
    ax1.set_title('1. Imagen Original', fontsize=15, fontweight='bold')
    ax1.axis('off')
    
    # Para pintar
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.imshow(img_para_pintar)
    ax2.set_title('2. PARA PINTAR (con números)', fontsize=15, fontweight='bold', color='blue')
    ax2.axis('off')
    
    # Solución
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.imshow(img_solucion)
    ax3.set_title('3. Solución (referencia de colores)', fontsize=15, fontweight='bold')
    ax3.axis('off')
    
    # Zoom de para pintar
    h, w = img_para_pintar.shape[:2]
    y1, y2 = h//3, 2*h//3
    x1, x2 = w//3, 2*w//3
    
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.imshow(img_para_pintar[y1:y2, x1:x2])
    ax4.set_title('4. Zoom - Detalle de números', fontsize=15, fontweight='bold')
    ax4.axis('off')
    
    # Paleta
    ax5 = fig.add_subplot(gs[2, :])
    ax5.imshow(paleta_numerada)
    ax5.set_title('5. Paleta de Colores Numerada', fontsize=15, fontweight='bold')
    ax5.axis('off')
    
    plt.tight_layout()
    plt.show()

def guardar_archivos_finales(img_para_pintar, img_solucion, paleta_numerada, 
                             df_regiones, prefijo="pintar_por_numeros"):
    """Guarda todos los archivos del proyecto"""
    
    print(f"\n{'='*70}")
    print("GUARDANDO ARCHIVOS FINALES")
    print(f"{'='*70}\n")
    
    # Crear carpeta de salida
    import os
    carpeta = "output_pintar_numeros"
    os.makedirs(carpeta, exist_ok=True)
    
    # Guardar imágenes
    archivos_guardados = []
    
    # 1. Imagen para pintar
    ruta = f"{carpeta}/{prefijo}_PARA_PINTAR.png"
    Image.fromarray(img_para_pintar).save(ruta, dpi=(300, 300))
    archivos_guardados.append(ruta)
    print(f"✓ {ruta}")
    
    # 2. Solución
    ruta = f"{carpeta}/{prefijo}_solucion.png"
    Image.fromarray(img_solucion).save(ruta, dpi=(300, 300))
    archivos_guardados.append(ruta)
    print(f"✓ {ruta}")
    
    # 3. Paleta
    ruta = f"{carpeta}/{prefijo}_paleta.png"
    Image.fromarray(paleta_numerada).save(ruta, dpi=(300, 300))
    archivos_guardados.append(ruta)
    print(f"✓ {ruta}")
    
    # 4. CSV con información de regiones
    ruta = f"{carpeta}/{prefijo}_regiones_info.csv"
    df_regiones.to_csv(ruta, index=False)
    archivos_guardados.append(ruta)
    print(f"✓ {ruta}")
    
    # 5. Versión de alta resolución para imprimir (opcional)
    ruta = f"{carpeta}/{prefijo}_PARA_PINTAR_alta_res.png"
    Image.fromarray(img_para_pintar).save(ruta, dpi=(600, 600), optimize=False)
    archivos_guardados.append(ruta)
    print(f"✓ {ruta}")
    
    print(f"\n{'='*70}")
    print(f"✓ TODOS LOS ARCHIVOS GUARDADOS EN: ./{carpeta}/")
    print(f"{'='*70}\n")
    
    return archivos_guardados


In [ ]:

# =============================================================================
# EJECUCIÓN DEL PASO 5 (FINAL)
# =============================================================================

print("="*70)
print("PASO 5: GENERAR CONTORNOS Y NÚMEROS (FINAL)")
print("="*70)

# Parámetros ajustables
GROSOR_BORDE = 3  # Grosor de los bordes en píxeles (2-5 recomendado)
TAMANO_NUMERO = 'auto'  # 'auto' o un número fijo (ej: 20, 30, 40)
FONDO_BLANCO = True  # True = fondo blanco, False = colores tenues de guía

print(f"\nParámetros:")
print(f"  - Grosor de borde: {GROSOR_BORDE}px")
print(f"  - Tamaño de número: {TAMANO_NUMERO}")
print(f"  - Fondo: {'Blanco' if FONDO_BLANCO else 'Colores tenues'}")

# Generar imagen final
img_para_pintar, img_solucion, paleta_numerada = generar_imagen_para_pintar(
    mapa_regiones_suavizado,
    df_regiones_suavizado,
    colores_paleta,
    grosor_borde=GROSOR_BORDE,
    tamano_numero=TAMANO_NUMERO,
    fondo_blanco=FONDO_BLANCO
)

# Visualizar resultado final
visualizar_resultado_final(
    imagen_original,
    img_para_pintar,
    img_solucion,
    paleta_numerada
)

# Guardar archivos
archivos = guardar_archivos_finales(
    img_para_pintar,
    img_solucion,
    paleta_numerada,
    df_regiones_suavizado,
    prefijo="mi_pintar_por_numeros"
)

# Resumen final
print("\n" + "="*70)
print("🎨 PROYECTO COMPLETADO - RESUMEN FINAL")
print("="*70)
print(f"Imagen original: {imagen_original.shape[0]}x{imagen_original.shape[1]} píxeles")
print(f"Colores utilizados: {len(colores_paleta)}")
print(f"Regiones totales: {len(df_regiones_suavizado)}")
print(f"Archivos generados: {len(archivos)}")
print(f"\nArchivos listos para:")
print(f"  ✓ Imprimir y pintar")
print(f"  ✓ Usar como referencia de colores")
print(f"  ✓ Compartir o procesar más")
print("="*70 + "\n")

print("🎉 ¡Listo para pintar! 🎨")